# NVIDIA DLI: Data Augmentation for ASL Recognition

**Source:** [NVIDIA Deep Learning Institute](https://www.nvidia.com/dli)

**Description:** This notebook demonstrates how to use Keras `ImageDataGenerator` to augment
training data for American Sign Language (ASL) letter recognition. Data augmentation reduces
overfitting and improves model generalization.

**Requirements:** TensorFlow/Keras, NumPy, Pandas, Matplotlib

**Dataset:** [Sign Language MNIST](https://www.kaggle.com/datamunge/sign-language-mnist)
- The dataset should be in `./data/asl_data/` with `sign_mnist_train.csv` and `sign_mnist_valid.csv`

## Objectives

- Augment the ASL dataset using Keras `ImageDataGenerator`
- Train an improved CNN model with augmented data
- Save the well-trained model to disk for deployment

## Setup and Data Preparation

In [ ]:
import tensorflow.keras as keras
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense, Conv2D, MaxPool2D, Flatten, Dropout, BatchNormalization,
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator

## Load and Prepare the Data

Load the ASL dataset from CSV files and reshape the images to 28x28 grayscale.

> **Note:** Update the paths below if your dataset is in a different location.

In [ ]:
# Load in our data from CSV files
train_df = pd.read_csv("data/asl_data/sign_mnist_train.csv")
valid_df = pd.read_csv("data/asl_data/sign_mnist_valid.csv")

# Separate out our target values
y_train = train_df['label']
y_valid = valid_df['label']
del train_df['label']
del valid_df['label']

# Separate our our image vectors
x_train = train_df.values
x_valid = valid_df.values

# Turn our scalar targets into binary categories
num_classes = 24
y_train = keras.utils.to_categorical(y_train, num_classes)
y_valid = keras.utils.to_categorical(y_valid, num_classes)

# Normalize our image data
x_train = x_train / 255
x_valid = x_valid / 255

# Reshape the image data for the convolutional network
x_train = x_train.reshape(-1,28,28,1)
x_valid = x_valid.reshape(-1,28,28,1)

## Model Creation

Build a CNN architecture with convolutional layers, batch normalization, max pooling,
dropout, and dense layers. This is the same architecture from previous lessons.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense,
    Conv2D,
    MaxPool2D,
    Flatten,
    Dropout,
    BatchNormalization,
)

model = Sequential()
model.add(Conv2D(75, (3, 3), strides=1, padding="same", activation="relu", 
                 input_shape=(28, 28, 1)))
model.add(BatchNormalization())
model.add(MaxPool2D((2, 2), strides=2, padding="same"))
model.add(Conv2D(50, (3, 3), strides=1, padding="same", activation="relu"))
model.add(Dropout(0.2))
model.add(BatchNormalization())
model.add(MaxPool2D((2, 2), strides=2, padding="same"))
model.add(Conv2D(25, (3, 3), strides=1, padding="same", activation="relu"))
model.add(BatchNormalization())
model.add(MaxPool2D((2, 2), strides=2, padding="same"))
model.add(Flatten())
model.add(Dense(units=512, activation="relu"))
model.add(Dropout(0.3))
model.add(Dense(units=num_classes, activation="softmax"))

## Data Augmentation

Keras `ImageDataGenerator` applies random transformations to training images on-the-fly:
- Random rotation (up to 10 degrees)
- Random zoom (up to 5%)
- Random horizontal shift (up to 10%)
- Random vertical shift (up to 10%)
- Horizontal flipping

> **Why no vertical flip?** Our dataset contains pictures of hands signing the alphabet.
> Hands are not typically seen upside-down, so vertical flipping would create unrealistic
> training samples.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=10,  # randomly rotate images in the range (degrees, 0 to 180)
    zoom_range=0.1,  # Randomly zoom image
    width_shift_range=0.1,  # randomly shift images horizontally (fraction of total width)
    height_shift_range=0.1,  # randomly shift images vertically (fraction of total height)
    horizontal_flip=True,  # randomly flip images horizontally
    vertical_flip=False, # Don't randomly flip images vertically
)  

## Batch Size and Visualization

The `ImageDataGenerator` batches the data, which improves training efficiency.
Let's visualize a batch of augmented images.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
batch_size = 32
img_iter = datagen.flow(x_train, y_train, batch_size=batch_size)

x, y = img_iter.next()
fig, ax = plt.subplots(nrows=4, ncols=8)
for i in range(batch_size):
    image = x[i]
    ax.flatten()[i].imshow(np.squeeze(image))
plt.show()

## Fit the Generator to Training Data

In [ ]:
datagen.fit(x_train)

## Compile the Model

Compile with categorical crossentropy loss for multi-class classification.

In [ ]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

## Train with Augmentation

When using an image data generator, we use `model.fit()` with the generator iterator
instead of raw data arrays. The `steps_per_epoch` parameter ensures we process the same
amount of data per epoch as without augmentation.

In [ ]:
model.fit(img_iter,
          epochs=10,
          steps_per_epoch=len(x_train)/batch_size, # Run same number of steps we would if we were not using a generator.
          validation_data=(x_valid, y_valid))

## Discussion of Results

The validation accuracy should be higher and more consistent compared to training without
augmentation. This indicates the model is no longer overfitting as severely, since the
augmented data provides more variation during training.

## Save the Model

Save the trained model to disk for later deployment and inference.

In [ ]:
model.save('asl_model')

## Summary

In this notebook we used Keras `ImageDataGenerator` to augment the ASL dataset. The result
is a trained model with less overfitting and improved validation accuracy, ready for
deployment to classify new hand sign images.